In [1]:
!pip install pytube youtube-transcript-api yt-dlp

  Using cached pytube-15.0.0-py3-none-any.whl.metadata (5.0 kB)
Using cached pytube-15.0.0-py3-none-any.whl (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [youtube-transcript-api]


In [1]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display , JSON
import yt_dlp
from typing import List , Dict
from langchain_core.messages import HumanMessage , ToolMessage
import json

import warnings
warnings.filterwarnings("ignore")
#Supress pytube errors
import logging
pytube_logger = logging.getLogger("pytube")
pytube_logger.setLevel(logging.ERROR)
# Supress yt-dlp warnings
yt_dlp_logger = logging.getLogger("yt_dlp")
yt_dlp_logger.setLevel(logging.ERROR)


In [2]:
from langchain.chat_models import init_chat_model
api_key = "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"
llm = init_chat_model(model = "llama-3.3-70b-versatile" , model_provider= "groq", api_key = api_key)

In [3]:
@tool
def extract_video_id(url:str)->str :
    """
    Extracts the 11 character Youtube video id from a url
    Parameters:
    -url(str): It is the youtube url which contains the youtube video id
    Returns:
    - str: Extracted youtube id or error message if parsing fails
    """
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern,url)
    return match.group(1) if match else "Error: Invalid youtube URL"
     
    

In [4]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11 character Youtube video id from a url
Parameters:
-url(str): It is the youtube url which contains the youtube video id
Returns:
- str: Extracted youtube id or error message if parsing fails
----------------------------
<function extract_video_id at 0x17d2b4f40>


In [5]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [6]:
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11 character Youtube video id from a url\nParameters:\n-url(str): It is the youtube url which contains the youtube video id\nReturns:\n- str: Extracted youtube id or error message if parsing fails', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x17d2b4f40>)

In [7]:
tools = []
tools.append(extract_video_id)

In [8]:
from youtube_transcript_api import YouTubeTranscriptApi
@tool
def fetch_transcript(video_id:str, language:str = "en")-> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id , languages = [language])
        return " ".join(snippet.text for snippet in transcript.snippets)
    except Exception as e:
        return f"Error : {str(e)}"

In [9]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [10]:
tools.append(fetch_transcript)

In [11]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
search_out=search_youtube.run("Generative AI")
display(JSON(search_out))

<IPython.core.display.JSON object>

In [13]:
tools.append(search_youtube)

In [16]:
@tool
def get_full_metadata(url:str)-> Dict :
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet':True, 'logger': yt_dlp_logger}) as ydl:
        info = ydl.extract_info(url , download= False)
        return{
            "title": info.get('title'),
            "views": info.get('view_count'),
            "duration": info.get("duration"),
            "channel": info.get("uploader"),
            "likes": info.get("like_count"),
            "dislikes":info.get("dislike_count"),
            "comments":info.get("comment_count"),
            "chapters": info.get("chapters" ,[])
        }

In [17]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

In [18]:
tools.append(get_full_metadata)

In [21]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dlp_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [22]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails))

<IPython.core.display.JSON object>

In [23]:
tools.append(get_thumbnails)

In [24]:
llm_with_tools = llm.bind_tools(tools)

In [25]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))
    

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [26]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


In [27]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]


In [28]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xej9z2jdn', 'function': {'arguments': '{"url":"https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'extract_video_id'}, 'type': 'function'}, {'id': '15mv8j98w', 'function': {'arguments': '{"language":"en","video_id":"T-D1OfcDW1M"}', 'name': 'fetch_transcript'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 910, 'total_tokens': 969, 'completion_time': 0.094306822, 'completion_tokens_details': None, 'prompt_time': 0.044374727, 'prompt_tokens_details': None, 'queue_time': 0.175840468, 'total_time': 0.138681549}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c3479-b0bf-7f33-8540-f3a4eef32502-0', tool_calls=[{'name': 'extract_video_id', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'xej9z2jdn', '

In [29]:
messages.append(response_1)

In [30]:
tool_mapping ={
    "get_thumbnails":get_thumbnails,
    "get_full_metadata":get_full_metadata,
    "extract_video_id": extract_video_id ,
    "fetch_transcript" : fetch_transcript ,
    "search_youtube" : search_youtube
}

In [31]:
tool_calls_1 = response_1.tool_calls
display(JSON(tool_calls_1))

<IPython.core.display.JSON object>

In [32]:
tool_name=tool_calls_1[0]['name']
print(tool_name)

extract_video_id


In [33]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

xej9z2jdn


In [34]:
args=tool_calls_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}


In [35]:
my_tool = tool_mapping[tool_calls_1[0]['name']]

In [36]:
video_id =my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [37]:
messages.append(ToolMessage(content = video_id, tool_call_id= tool_calls_1[0]['id']))

In [38]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9ab56t9ff', 'function': {'arguments': '{"video_id":"T-D1OfcDW1M"}', 'name': 'fetch_transcript'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 987, 'total_tokens': 1011, 'completion_time': 0.055517743, 'completion_tokens_details': None, 'prompt_time': 0.050809756, 'prompt_tokens_details': None, 'queue_time': 0.171112611, 'total_time': 0.106327499}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c3481-d26d-7f83-b17f-f052e73be8fc-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'video_id': 'T-D1OfcDW1M'}, 'id': '9ab56t9ff', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 987, 'output_tokens': 24, 'total_tokens': 1011})

In [39]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M'},
  'id': '9ab56t9ff',
  'type': 'tool_call'}]

In [40]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

In [41]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [42]:
summary = llm_with_tools.invoke(messages)

In [43]:
summary

AIMessage(content='The video discusses the challenges of large language models (LLMs) and how they can be improved using a framework called Retrieval-Augmented Generation (RAG). The speaker, Marina Danilevsky, explains that LLMs can generate text in response to a user query, but they often lack sources and can be out of date. She illustrates this with an anecdote about her kids asking her about the planet with the most moons, and how she confidently gave an incorrect answer.\n\nThe speaker then explains how RAG works by adding a content store to the LLM, which allows it to retrieve relevant information before generating a response. This approach helps to address the two challenges of LLMs: being out of date and lacking sources. The RAG framework instructs the LLM to first retrieve relevant content and then combine it with the user\'s question to generate a response.\n\nThe speaker also discusses the benefits of RAG, including being able to provide evidence for the response and knowing 

In [44]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

In [45]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [46]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 The YouTube video is about LangChain, an open-source orchestration framework for developing applications that use large language models. It provides a centralized development environment to build and integrate language model applications with data sources and software workflows. LangChain streamlines the programming of language model applications through abstractions, which represent common steps and concepts necessary to work with language models. The framework includes components such as the LLM module, prompts, chains, indexes, and agents, which can be used to create applications that use language models to perform tasks such as summarization, question answering, and data augmentation. LangChain is free to use and has related frameworks like LangServe and LangSmith for creating chains as REST APIs and monitoring, evaluating, and debugging applications.


In [47]:
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

In [48]:
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

In [49]:
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
)

In [50]:
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

In [51]:
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
)

In [52]:
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"])

In [53]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [54]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = summarization_chain.invoke(query)
print("Video Summary:\n", result)

Video Summary:
 


In [55]:
query = {"query": "Get top 3 youtube videos in India and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Non-critical network error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_youtube{"query": "top youtube videos in India"}</function>\n'}}


In [56]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

In [57]:
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]

In [58]:
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))

In [59]:
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

In [60]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [61]:
query_us = {"query": "Show top 3 US trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)

ERROR: [youtube:tab] trending: The channel/playlist does not exist and the URL redirected to youtube.com home page
ERROR: [youtube:tab] trending: The channel/playlist does not exist and the URL redirected to youtube.com home page



US Trending Videos:
 content='Here are the top 3 US trending videos with metadata and thumbnails:\n\n1. **US News LIVE: Trump SHAKES The World With BIGGEST War Warning , Is World War III On The Brink?**\n\t* Video ID: R0o8WKz5IAY\n\t* URL: https://youtu.be/R0o8WKz5IAY\n\t* Metadata: \n\t* Thumbnails: \n2. **5 Most Profitable Niches on YouTube in 2025**\n\t* Video ID: Ufqc0lU-i84\n\t* URL: https://youtu.be/Ufqc0lU-i84\n\t* Metadata: \n\t* Thumbnails: \n3. **Genderbend (Among Us Animation meme) #trending Original song by @phuongmychiofficial**\n\t* Video ID: aGC4tONT9QA\n\t* URL: https://youtu.be/aGC4tONT9QA\n\t* Metadata: \n\t* Thumbnails: \n\nNote: The metadata and thumbnails for each video are not available due to the error in the previous response.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 230, 'prompt_tokens': 1661, 'total_tokens': 1891, 'completion_time': 0.427158495, 'completion_tokens_details': None, 'prompt_time': 0.534581831, 'prompt_tokens_